# Subagents and scope

An agent can call `invoke` from inside its own code. That makes a *tree* of agents — which is how you decompose a job too big for one REPL session.

It also raises a question the first two notebooks left alone: if inputs are variables in one agent's REPL, what does a **subagent** see? The answer is the point of this notebook, and it is not 'everything'.

## Setup

Install JAZ, provide an API key, and pick a model. Everything below runs in Colab — no local install.

In [1]:
%pip install -q jaz-lang

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [3]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## An agent calling agents

`invoke` is already available inside the REPL — nothing to import or pass. Here the top-level agent farms each ticket out to its own subagent.

Watch the `depth=` field in the log. `depth=1` is the agent you called; `depth=2` is a subagent it spawned. That is how you know recursion actually happened, rather than the agent quietly doing everything itself.

In [4]:
from jaz import invoke
from jaz.hooks import ReturnType, PrintLogger

with PrintLogger(max_field_length=90):
    categories = invoke(
        ReturnType(list),
        task=(
            "For each ticket in tickets, call invoke(ReturnType(str), task=..., ticket=...) "
            "to get a one-word category for it. Collect the results and return the list."
        ),
        tickets=[
            "Card declined at checkout",
            "How do I export my data?",
            "App crashes on launch",
        ],
    )

categories

2026-09-15 17:03:08 [Agent] invoke enter: inputs={'task': 'For each ticket in tickets, call invoke(ReturnType(str), task=..., ticket=...) t...[truncated], depth=1, hooks=[PrintLogger(), ReturnType()]


2026-09-15 17:03:08 [Agent] invoke send: inputs={'task': 'For each ticket in tickets, call invoke(ReturnType(str), task=..., ticket=...) t...[truncated], depth=1


2026-09-15 17:03:08 [Agent] message: {'role': 'system', 'content': 'You are in a multiturn REPL session.\n\n<response_format>\n- Respond with ONLY code. Your entire response will be sent verbatim to the REPL and executed as code.\n  Anything in your response that is not valid code is a syntax error.\n  - All natural language prose MUST be written as *comments* in your code.\n  - Do NOT wrap your code in markdown fences or XML tags.\n- Write a brief plan for your next step in comments on the first few lines of your response/code:\n\n    # <brief plan for next step>\n    code_for_next_step\n\n</response_format>\n\n<repl_spec>\n## Python REPL specification\n\n- Your response should contain Python code.\n- After the REPL executes your code, the printed output including any errors will be shown\n  to you in the next iteration. It is NOT shown to the caller.\n- Printed output will be truncated if it is too large.\n- The REPL state persists: if you assign a variable, it will be available in f

2026-09-15 17:03:08 [Agent] message: {'role': 'user', 'content': '<task type="str">\nFor each ticket in tickets, call invoke(ReturnType(str), task=..., ticket=...) to get a one-word category for it. Collect the results and return the list.\n</task>\n<tickets type="list">\n[\'Card declined at checkout\', \'How do I export my data?\', \'App crashes on launch\']\n</tickets>\nThe input variables `task`, `tickets` are available in your REPL with descriptions given above.\nThey are not automatically available to `invoke()` calls you make, so making them available to sub-invokes requires passing them explicitly.\n'}


2026-09-15 17:03:08 [LLM] query enter: model=openai/gpt-5.4-nano, messages=2 messages


...

2026-09-15 17:03:27 [Agent] repl iteration 0 output: exec_result=Return(return_value=[[['Need more details to diagnose: please provide the crash log/stack ...[truncated], depth=2


2026-09-15 17:03:27 [Agent] invoke exit: result=Return(return_value=[[['Need more details to diagnose: please provide the crash log/stack ...[truncated], depth=2


2026-09-15 17:03:27 [Agent] repl iteration 2 output: exec_result=Return(return_value=[['Payments'], ['To export your data, look for an option such as “Expo...[truncated], depth=1


2026-09-15 17:03:27 [Agent] invoke exit: result=Return(return_value=[['Payments'], ['To export your data, look for an option such as “Expo...[truncated], depth=1


[['Payments'],
 ['To export your data, look for an option such as “Export”, “Download”, or “Data” in the product’s settings/account menu. Choose the format you need (e.g., CSV/Excel/JSON), select the date range or fields, then confirm to download. If there’s no export button, check for “API”, “Integrations”, or “Admin” tools—many systems let you export via an API endpoint or admin console. If you tell me which app/service you’re using, I can give exact click-by-click steps.'],
 [[['Need more details to diagnose: please provide the crash log/stack trace, device/OS (and app version), and steps that reproduce the crash (e.g., just launch vs. after login). If possible, paste the first ~50 lines around the error.']]]]

## Inputs stop at the call that received them

Here is the part that surprises people. An input you pass to `invoke` is bound in **that agent's** REPL and nowhere else. Subagents do not inherit it.

To check that honestly we read the prompts themselves with a small hook, rather than asking an agent to tell us what it can see:

In [5]:
from jaz.hooks import Hook


class CapturePrompts(Hook):
    """Records the full prompt of every invoke in the tree.

    We read the prompts themselves rather than asking an agent to report what it can see —
    the prompt is ground truth, an agent's description of it is not.
    """

    def __init__(self):
        self.prompts = []

    def on_llm_query_enter(self, event):
        # Every message, not just the user one: a per-call input is described in the user
        # message, a scoped value in the system message.
        whole = "\n".join(m["content"] for m in event.messages)
        if whole not in self.prompts:
            self.prompts.append(whole)
        return []


def stock_level(sku: str) -> int:
    """Units currently in stock for a SKU."""
    return {"SKU-1": 12, "SKU-2": 0}[sku]


TASK = (
    "Call invoke(ReturnType(int), task='Return the number 1.') and return its result."
)

In [6]:
seen = CapturePrompts()

with seen:
    invoke(ReturnType(int), task=TASK, stock_level=stock_level)

for i, prompt in enumerate(seen.prompts):
    print(f"invoke #{i}  mentions stock_level: {'stock_level' in prompt}")

invoke #0  mentions stock_level: True
invoke #1  mentions stock_level: False


The top-level invoke has `stock_level`. The subagent's prompt does not mention it at all — the name is simply not bound down there.

That is deliberate: an agent's inputs are its own, and a deep tree does not silently accumulate every caller's variables. But it means passing a tool the ordinary way gives it to exactly one agent.

## `scope` — inputs for a whole subtree

`scope` binds values as **ambient inputs**: everything inside the `with` block sees them, nested invokes included, all the way down.

Same code, one line different:

In [7]:
from jaz import scope

seen = CapturePrompts()

with seen, scope(stock_level=stock_level):
    invoke(ReturnType(int), task=TASK)

for i, prompt in enumerate(seen.prompts):
    print(f"invoke #{i}  mentions stock_level: {'stock_level' in prompt}")

invoke #0  mentions stock_level: True
invoke #1  mentions stock_level: True


Both prompts now carry it, and nothing else changed.

One detail worth knowing: a per-call input is described in the **user** message, while a scoped value is described in the **system** message — it is ambient, so it belongs to the session rather than to one request.

This is why `scope` is the usual way to give an agent its tools: most tools should be available to the whole tree, not to the top call alone. It is equivalent to passing the value to every `invoke` beneath it by hand — without the hand-threading.

Now a real use of it — subagents that actually call the scoped tool:

In [8]:
from jaz.hooks import ValidateReturn


def two_skus(result):
    assert set(result) == {"SKU-1", "SKU-2"}, f"expected both SKUs, got {sorted(result)}"
    assert all(isinstance(v, int) for v in result.values()), "values must be ints"


with scope(stock_level=stock_level):
    stock = invoke(
        ReturnType(dict),
        ValidateReturn(two_skus),
        task=(
            "For each of SKU-1 and SKU-2, call invoke(ReturnType(int), task=...) and have "
            "the subagent call stock_level to look the value up. Return a flat dict "
            "mapping each sku string to its integer stock level, e.g. {'SKU-9': 3}."
        ),
    )

stock

{'SKU-1': 12, 'SKU-2': 0}

## Capping the depth

Recursion is unbounded by default. `RecursionLimit` caps how deep the tree may grow, and because it is a context manager the cap reaches the whole subtree.

It is worth seeing *how* it enforces that. At `max_depth=1` the agent is not given an `invoke` to call at all — the capability is absent from its REPL, rather than blowing up when used:

In [9]:
from jaz.hooks import RecursionLimit

question = "Return True if the name invoke is defined in your REPL, else return False."

print("no limit    ->", invoke(ReturnType(bool), task=question))

with RecursionLimit(max_depth=1):
    print("max_depth=1 ->", invoke(ReturnType(bool), task=question))

no limit    -> True


max_depth=1 -> False


## What to take away

- **Subagents are just `invoke` from inside the REPL.** The `depth=` field in a log is how you confirm the tree is real.
- **Inputs do not propagate.** They belong to the agent that received them.
- **`scope` is how you reach the subtree.** Bind tools and shared data there, not as per-call inputs.
- **`RecursionLimit` bounds the tree** by withholding `invoke`, and applies to everything inside its `with` block.

---

**Next:** [Budgets and failure](04-budgets-and-failure.ipynb) — what happens when a run costs too much, or the agent returns something wrong.